In [ ]:
import os
import time
import numpy as np
import torch

import matplotlib

# 非交互式后端：保存图片但不弹窗
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from ase import Atoms
from ase.calculators.calculator import Calculator, all_changes

try:
    from ase.mep.neb import NEB, NEBOptimizer
except ImportError:
    from ase.neb import NEB, NEBOptimizer


# ============================================================
# 复用现有 alanine restrained-MD oracle 和绘图函数
#
# 必须确保 IVM_shell.py 最后使用：
#
# if __name__ == "__main__":
#     main()
#
# 否则 import 时会直接启动 IVM 网格搜索。
# ============================================================
import IVM_shell as ala_ivm

from ase_mfep_neb import (
    install_central_difference_improved_tangent,
)


M_gradient = ala_ivm.M_gradient
fig_save = ala_ivm.fig_save

# 关闭 GROMACS 大量终端输出
if hasattr(ala_ivm, "GROMACS_QUIET"):
    ala_ivm.GROMACS_QUIET = True


# ============================================================
# 基本设置
# ============================================================
device = torch.device("cpu")
dtype = torch.float64

seed = 41

np.random.seed(seed)
torch.manual_seed(seed)

# 与 IVM 完全相同的端点
C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

# 内部 images 数，与 IVM 相同
N = 20

xlim = (-np.pi, 0.0)
ylim = (0.0, np.pi)

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)


# ============================================================
# Restrained-MD 参数
# 与 IVM 中 gradient_md_use 的默认设置一致
# ============================================================
DISCARD_FRAC = 0.05
TMIN = None
TMAX = None


# ============================================================
# NEB / ODE 参数
# ============================================================

# 不向 NEB 传入 k，因此使用 ASE 默认 k=0.1
ASE_DEFAULT_K = 0.1

# 最大 accepted ODE steps
MAX_NEB_STEPS = 200

# ODE 终止条件
NEB_FMAX_TOL = 1e-8

# 是否强制覆盖已经完成的结果
FORCE_RERUN = True


# ============================================================
# ASE ODE force cap
#
# ASE ode12r 默认 maxtol=1000。
#
# 这里只限制传给 ASE 的 |M grad F|。
# 原始 M、grad F、M grad F 仍用于诊断和指标计算。
#
# 当 |M grad F| <= ASE_FORCE_CAP 时，力完全不变。
# 当 |M grad F| > ASE_FORCE_CAP 时，只缩小模长，不改变方向。
# ============================================================
ASE_FORCE_CAP = 500.0

# True：每次发生 force cap 时打印信息。
# False：不逐 image 打印，减少 terminal 输出。
PRINT_FORCE_CAPPING = False


# ============================================================
# 图片与保存设置
# ============================================================

# 保存 path 图片，但不显示
SAVE_FIGS = True

# 每隔多少个 accepted ODE steps 保存一次图片
SAVE_FIG_EVERY = 10

# 每个 accepted ODE state 都写入 npz
SAVE_EVERY_STATE = True


# ============================================================
# 全局 restrained-MD oracle 计数
#
# 每个 internal image 真正调用一次 M_gradient：
#
#     GLOBAL_IMAGE_MD_CALLS += 1
#
# 这是所有内部 images 的调用次数总和。
# ============================================================
GLOBAL_IMAGE_MD_CALLS = 0


# ============================================================
# CV <-> ASE Atoms
#
# 使用一个虚拟 H 原子存储二维 CV：
#
#     position[0,0] = phi
#     position[0,1] = psi
#     position[0,2] = 0
# ============================================================
def atoms_to_q(atoms):
    positions = atoms.get_positions()

    return np.array(
        [
            positions[0, 0],
            positions[0, 1],
        ],
        dtype=float,
    )


def make_cv_atoms(q, calculator=None):
    q = np.asarray(
        q,
        dtype=float,
    ).reshape(2,)

    atoms = Atoms(
        "H",
        positions=[
            [
                q[0],
                q[1],
                0.0,
            ]
        ],
    )

    atoms.pbc = False

    if calculator is not None:
        atoms.calc = calculator

    return atoms


# ============================================================
# 端点 calculator
#
# 端点固定，不进行 restrained MD。
# ============================================================
class ZeroEndpointCalculator(Calculator):
    implemented_properties = [
        "energy",
        "forces",
    ]

    def calculate(
        self,
        atoms=None,
        properties=("energy",),
        system_changes=all_changes,
    ):
        super().calculate(
            atoms,
            properties,
            system_changes,
        )

        self.results["energy"] = 0.0

        self.results["forces"] = np.zeros(
            (len(atoms), 3),
            dtype=float,
        )


# ============================================================
# Alanine MFEP calculator
#
# 原始 oracle：
#
#     M(q)
#     grad F(q)
#
# 原始 Riemannian gradient：
#
#     Mg_raw = M(q) grad F(q)
#
# 传给 ASE 的 force：
#
#     force_ASE = -Mg_ase
#
# 其中 Mg_ase 是对 Mg_raw 做模长截断后的向量：
#
#     Mg_ase = Mg_raw,                     |Mg_raw| <= cap
#
#     Mg_ase = cap * Mg_raw / |Mg_raw|,   |Mg_raw| > cap
#
# 重要：
#   last_Mg 保存未截断的 Mg_raw，因此两个积分指标仍然
#   使用真实的 M grad F，与 IVM 完全一致。
# ============================================================
class AlanineMFEPGradientCalculator(Calculator):
    implemented_properties = [
        "energy",
        "forces",
    ]

    def __init__(
        self,
        discard_frac=0.05,
        tmin=None,
        tmax=None,
        force_cap=ASE_FORCE_CAP,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.discard_frac = discard_frac
        self.tmin = tmin
        self.tmax = tmax

        self.force_cap = (
            None
            if force_cap is None
            else float(force_cap)
        )

        if (
            self.force_cap is not None
            and self.force_cap <= 0.0
        ):
            raise ValueError(
                f"force_cap must be positive or None, "
                f"got {self.force_cap}"
            )

        # 该 image 真正调用 M_gradient 的累计次数
        self.md_calls = 0

        # 最近一次 oracle 的坐标和原始数据
        self.last_q = None
        self.last_M = None
        self.last_F = None

        # 原始、未截断的 M grad F
        # 用于计算正确的路径指标
        self.last_Mg = None

        # 实际传给 ASE 的截断后 M grad F
        self.last_Mg_ase = None

        # 截断诊断
        self.last_raw_Mg_norm = None
        self.last_ase_Mg_norm = None
        self.last_force_scale = None
        self.last_force_was_capped = None

    def calculate(
        self,
        atoms=None,
        properties=("energy",),
        system_changes=all_changes,
    ):
        global GLOBAL_IMAGE_MD_CALLS

        super().calculate(
            atoms,
            properties,
            system_changes,
        )

        q = atoms_to_q(atoms)

        q_tensor = torch.tensor(
            q.reshape(1, 2),
            dtype=dtype,
            device=device,
        )

        # ----------------------------------------------------
        # M_gradient 返回：
        #
        # M:
        #     shape (2, 2)
        #
        # F_orig:
        #     shape (2,)
        #     原始单位 kJ/mol/rad
        # ----------------------------------------------------
        M, F_orig = M_gradient(
            q_tensor,
            discard_frac=self.discard_frac,
            tmin=self.tmin,
            tmax=self.tmax,
        )

        M = np.asarray(
            M,
            dtype=float,
        ).reshape(2, 2)

        F_orig = np.asarray(
            F_orig,
            dtype=float,
        ).reshape(2,)

        # ----------------------------------------------------
        # 数值检查
        # ----------------------------------------------------
        if not np.all(np.isfinite(M)):
            raise FloatingPointError(
                f"Non-finite M at q={q}:\n{M}"
            )

        if not np.all(np.isfinite(F_orig)):
            raise FloatingPointError(
                f"Non-finite F_orig at q={q}: "
                f"{F_orig}"
            )

        # ----------------------------------------------------
        # 与 IVM 一致：
        # kJ/mol -> kcal/mol
        # ----------------------------------------------------
        F = F_orig / 4.184

        # ----------------------------------------------------
        # 原始 Riemannian gradient：
        #
        # Mg_raw = M grad F
        #
        # 此值不截断，用于保存和指标计算。
        # ----------------------------------------------------
        Mg_raw = M @ F

        if not np.all(np.isfinite(Mg_raw)):
            raise FloatingPointError(
                f"Non-finite M @ F at q={q}:\n"
                f"M = {M}\n"
                f"F = {F}\n"
                f"M @ F = {Mg_raw}"
            )

        raw_Mg_norm = float(
            np.linalg.norm(Mg_raw)
        )

        # ----------------------------------------------------
        # 只对传给 ASE 的向量做模长截断
        #
        # 小于上限时：
        #     force_scale = 1
        #     Mg_ase = Mg_raw
        #
        # 超过上限时：
        #     force_scale = cap / |Mg_raw|
        #     Mg_ase = force_scale * Mg_raw
        #
        # 方向和零点均保持不变。
        # ----------------------------------------------------
        force_scale = 1.0
        force_was_capped = False

        if (
            self.force_cap is not None
            and raw_Mg_norm > self.force_cap
        ):
            force_scale = (
                self.force_cap
                / (raw_Mg_norm + 1e-16)
            )

            force_was_capped = True

        Mg_ase = (
            force_scale
            * Mg_raw
        )

        ase_Mg_norm = float(
            np.linalg.norm(Mg_ase)
        )

        if not np.all(np.isfinite(Mg_ase)):
            raise FloatingPointError(
                f"Non-finite capped M @ F at q={q}: "
                f"{Mg_ase}"
            )

        if (
            self.force_cap is not None
            and ase_Mg_norm
            > self.force_cap * (1.0 + 1e-12)
        ):
            raise RuntimeError(
                "ASE force cap failed: "
                f"raw norm={raw_Mg_norm}, "
                f"ASE norm={ase_Mg_norm}, "
                f"cap={self.force_cap}"
            )

        if (
            PRINT_FORCE_CAPPING
            and force_was_capped
        ):
            print(
                "[ASE force cap] "
                f"q=({q[0]:.8f}, {q[1]:.8f})  "
                f"raw_norm={raw_Mg_norm:.6e}  "
                f"cap={self.force_cap:.6e}  "
                f"scale={force_scale:.6e}  "
                f"ase_norm={ase_Mg_norm:.6e}"
            )

        # ----------------------------------------------------
        # ASE force
        #
        # ASE 使用 force = -gradient：
        #
        #     force_ASE = -Mg_ase
        # ----------------------------------------------------
        ase_forces = np.zeros(
            (len(atoms), 3),
            dtype=float,
        )

        ase_forces[0, 0] = -Mg_ase[0]
        ase_forces[0, 1] = -Mg_ase[1]

        # 没有自由能值，用常数零占位
        self.results["energy"] = 0.0
        self.results["forces"] = ase_forces

        # ----------------------------------------------------
        # 保存最近一次数据
        # ----------------------------------------------------
        self.last_q = q.copy()
        self.last_M = M.copy()
        self.last_F = F.copy()

        # 原始、未截断值：
        # 用于正确的 IVM/NEB 对比指标
        self.last_Mg = Mg_raw.copy()

        # 实际传给 ASE 的值
        self.last_Mg_ase = Mg_ase.copy()

        self.last_raw_Mg_norm = raw_Mg_norm
        self.last_ase_Mg_norm = ase_Mg_norm
        self.last_force_scale = force_scale
        self.last_force_was_capped = force_was_capped

        # 当前 image 的独立计数器
        self.md_calls += 1

        # 所有 images 的全局总计数器
        GLOBAL_IMAGE_MD_CALLS += 1


# ============================================================
# 与 IVM 相同的直线初始路径
# ============================================================
def get_linear_initial_path(
    q0,
    q1,
    n_internal,
):
    t = np.linspace(
        0.0,
        1.0,
        n_internal + 2,
    )[1:-1]

    return (
        (1.0 - t[:, None]) * q0[None, :]
        + t[:, None] * q1[None, :]
    )


# ============================================================
# 从 ASE images 提取完整二维路径
# ============================================================
def images_to_path(images):
    return np.array(
        [
            atoms_to_q(image)
            for image in images
        ],
        dtype=float,
    )


# ============================================================
# 与 IVM 相同的中心差分 tangent
#
# path_full: (N+2,2)
# tangent:   (N,2)
# ============================================================
def central_difference_tangent(path_full):
    path_full = np.asarray(
        path_full,
        dtype=float,
    )

    return (
        path_full[2:]
        - path_full[:-2]
    )


# ============================================================
# 构造 G-正交投影
#
# G = M^{-1}
# ============================================================
def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_images = tangent.shape[0]

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_images, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_images, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_images):
        v = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v @ G_i @ v
        )

        norm_G = np.sqrt(
            max(norm_G_sq, 1e-16)
        )

        tau_G = v / norm_G
        G_tau_G = G_i @ tau_G

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-16)
        )

        P_parallel_all[i] = P_parallel
        P_perp_all[i] = identity - P_parallel

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


# ============================================================
# Euclidean normal projection
#
# 用于记录 ASE NEB 实际使用的：
#
#     P_E (M grad F)
# ============================================================
def compute_Euclidean_perp(
    vectors,
    tangent,
):
    vectors = np.asarray(
        vectors,
        dtype=float,
    )

    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    tangent_norm = np.linalg.norm(
        tangent,
        axis=1,
        keepdims=True,
    )

    tau = tangent / (
        tangent_norm + 1e-16
    )

    inner = np.sum(
        tau * vectors,
        axis=1,
        keepdims=True,
    )

    return (
        vectors
        - inner * tau
    )


# ============================================================
# 计算与 IVM 一致的两个整路径指标
#
# 1. 论文中的 G-metric I_perp：
#
#    I_perp_G
#    = int |P_G M grad F|_G^2 |phi'|_G ds
#
# 2. Euclidean residual integral：
#
#    I_perp_E
#    = int ||P_G M grad F||_2^2 |phi'|_2 ds
# ============================================================
def compute_integrated_metrics(
    path_full,
    M_all,
    F_all,
):
    path_full = np.asarray(
        path_full,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    path_internal = path_full[1:-1]

    tangent = central_difference_tangent(
        path_full
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    # 原始、未截断的 M grad F
    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    # G-normal component
    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    # Euclidean-normal component
    perpE_Mg = compute_Euclidean_perp(
        Mg_all,
        tangent,
    )

    # 相邻 segments
    seg_minus = (
        path_internal
        - path_full[:-2]
    )

    seg_plus = (
        path_full[2:]
        - path_internal
    )

    # --------------------------------------------------------
    # G-arclength quadrature weights
    # --------------------------------------------------------
    seg_minus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    seg_plus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    seg_minus_G = np.sqrt(
        np.maximum(
            seg_minus_G_sq,
            0.0,
        )
    )

    seg_plus_G = np.sqrt(
        np.maximum(
            seg_plus_G_sq,
            0.0,
        )
    )

    node_weights_G = 0.5 * (
        seg_minus_G
        + seg_plus_G
    )

    # --------------------------------------------------------
    # Euclidean arclength quadrature weights
    # --------------------------------------------------------
    seg_minus_E = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    seg_plus_E = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    node_weights_E = 0.5 * (
        seg_minus_E
        + seg_plus_E
    )

    # --------------------------------------------------------
    # G-metric residual
    # --------------------------------------------------------
    perpG_norm_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    perpG_norm_G_sq = np.maximum(
        perpG_norm_G_sq,
        0.0,
    )

    I_perp_G = float(
        np.sum(
            perpG_norm_G_sq
            * node_weights_G
        )
    )

    # --------------------------------------------------------
    # Euclidean residual integral
    # --------------------------------------------------------
    perpG_norm_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    I_perp_E = float(
        np.sum(
            perpG_norm_E_sq
            * node_weights_E
        )
    )

    # 与当前 IVM fmax 定义一致
    fmax_Gperp_Euclidean = float(
        np.max(
            np.sqrt(
                perpG_norm_E_sq
            )
        )
    )

    # ASE NEB 实际物理法向力的未截断诊断
    fmax_NEB_physical = float(
        np.max(
            np.linalg.norm(
                perpE_Mg,
                axis=1,
            )
        )
    )

    return {
        "G_all": G_all,
        "P_parallel_G_all": P_parallel_G_all,
        "P_perp_G_all": P_perp_G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "perpE_Mg": perpE_Mg,
        "node_weights_G": node_weights_G,
        "node_weights_E": node_weights_E,
        "perpG_norm_G_sq": perpG_norm_G_sq,
        "perpG_norm_E_sq": perpG_norm_E_sq,
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,
        "fmax_Gperp_Euclidean": (
            fmax_Gperp_Euclidean
        ),
        "fmax_NEB_physical": (
            fmax_NEB_physical
        ),
    }


# ============================================================
# 收集当前 internal images 的 M、F 和原始 M F
#
# 调用前必须先调用 neb.get_forces()。
# ============================================================
def collect_internal_oracle_data(images):
    M_list = []
    F_list = []
    Mg_list = []

    for image_index, image in enumerate(
        images[1:-1],
        start=1,
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineMFEPGradientCalculator,
        ):
            raise TypeError(
                f"Internal image {image_index} does not use "
                f"AlanineMFEPGradientCalculator."
            )

        if (
            calculator.last_q is None
            or calculator.last_M is None
            or calculator.last_F is None
            or calculator.last_Mg is None
        ):
            raise RuntimeError(
                f"No oracle data available for image "
                f"{image_index}. Call neb.get_forces() first."
            )

        current_q = atoms_to_q(
            image
        )

        if not np.allclose(
            current_q,
            calculator.last_q,
            rtol=0.0,
            atol=1e-12,
        ):
            raise RuntimeError(
                f"Cached oracle position for image "
                f"{image_index} is stale."
            )

        M_list.append(
            calculator.last_M.copy()
        )

        F_list.append(
            calculator.last_F.copy()
        )

        Mg_list.append(
            calculator.last_Mg.copy()
        )

    return (
        np.stack(
            M_list,
            axis=0,
        ),
        np.stack(
            F_list,
            axis=0,
        ),
        np.stack(
            Mg_list,
            axis=0,
        ),
    )


# ============================================================
# 获取所有内部 images 各自的 force/oracle 调用次数
#
# count_seq 直接采用第一个内部 image 的调用次数减去 1。
# ============================================================
def get_internal_image_md_calls(images):
    calls_all = []

    for image_index, image in enumerate(
        images[1:-1],
        start=1,
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineMFEPGradientCalculator,
        ):
            raise TypeError(
                f"Internal image {image_index} does not use "
                f"AlanineMFEPGradientCalculator."
            )

        calls_all.append(
            int(calculator.md_calls)
        )

    calls_all = np.asarray(
        calls_all,
        dtype=int,
    )

    if calls_all.size == 0:
        raise RuntimeError(
            "No interior images found."
        )

    first_image_calls = int(
        calls_all[0]
    )

    return (
        calls_all,
        first_image_calls,
    )


# ============================================================
# 收集 force-cap 诊断信息
# ============================================================
def collect_internal_force_cap_data(images):
    Mg_ase_list = []
    force_scale_list = []
    force_capped_list = []
    raw_norm_list = []
    ase_norm_list = []

    for image_index, image in enumerate(
        images[1:-1],
        start=1,
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineMFEPGradientCalculator,
        ):
            raise TypeError(
                f"Internal image {image_index} does not use "
                f"AlanineMFEPGradientCalculator."
            )

        if (
            calculator.last_Mg_ase is None
            or calculator.last_force_scale is None
            or calculator.last_force_was_capped is None
            or calculator.last_raw_Mg_norm is None
            or calculator.last_ase_Mg_norm is None
        ):
            raise RuntimeError(
                f"No force-cap data available for image "
                f"{image_index}. Call neb.get_forces() first."
            )

        Mg_ase_list.append(
            calculator.last_Mg_ase.copy()
        )

        force_scale_list.append(
            calculator.last_force_scale
        )

        force_capped_list.append(
            calculator.last_force_was_capped
        )

        raw_norm_list.append(
            calculator.last_raw_Mg_norm
        )

        ase_norm_list.append(
            calculator.last_ase_Mg_norm
        )

    return (
        np.stack(
            Mg_ase_list,
            axis=0,
        ),
        np.asarray(
            force_scale_list,
            dtype=float,
        ),
        np.asarray(
            force_capped_list,
            dtype=bool,
        ),
        np.asarray(
            raw_norm_list,
            dtype=float,
        ),
        np.asarray(
            ase_norm_list,
            dtype=float,
        ),
    )


# ============================================================
# 创建 NEB images
# ============================================================
def build_neb_images():
    initial_internal = get_linear_initial_path(
        C1,
        C2,
        N,
    )

    images = []

    # 固定起点
    images.append(
        make_cv_atoms(
            C1,
            calculator=ZeroEndpointCalculator(),
        )
    )

    # 内部 images
    for q in initial_internal:
        images.append(
            make_cv_atoms(
                q,
                calculator=(
                    AlanineMFEPGradientCalculator(
                        discard_frac=DISCARD_FRAC,
                        tmin=TMIN,
                        tmax=TMAX,
                        force_cap=ASE_FORCE_CAP,
                    )
                ),
            )
        )

    # 固定终点
    images.append(
        make_cv_atoms(
            C2,
            calculator=ZeroEndpointCalculator(),
        )
    )

    return images


# ============================================================
# 结果路径
# ============================================================
def neb_result_paths():
    casenum = (
        f"MFEP_ASE_NEB_ODE_N{N}"
        f"_kDefault"
        f"_forceCap{ASE_FORCE_CAP:g}"
        f"_centralImprovedTangent"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    figure_path = os.path.join(
        folder_path,
        "figures",
    )

    result_file = os.path.join(
        folder_path,
        f"alanine_{casenum}.npz",
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    return (
        casenum,
        folder_path,
        figure_path,
        result_file,
        done_file,
    )

class VerboseNEBOptimizer(NEBOptimizer):
    def run_ode(self, fmax):
        from ase.optimize.ode import ode12r

        try:
            ode12r(
                self.force_function,
                self.neb.get_positions().reshape(-1),
                fmax=fmax,
                rtol=self.rtol,
                C1=self.C1,
                C2=self.C2,
                steps=self.max_steps,
                verbose=self.verbose,
                callback=self.callback,
                residual=self.get_residual,
            )
            return True

        except Exception as exc:
            print("\n[NEB-ODE stopped]")
            print(f"Exception type : {type(exc).__name__}")
            print(f"Reason         : {exc}")
            print(f"Accepted steps : {self.nsteps}")
            print(
                f"Current residual: "
                f"{self.neb.get_residual():.6e}"
            )
            return False
# ============================================================
# 运行唯一一次 NEB ODE 实验
# ============================================================
def run_neb_experiment():
    global GLOBAL_IMAGE_MD_CALLS

    GLOBAL_IMAGE_MD_CALLS = 0

    (
        casenum,
        folder_path,
        figure_path,
        result_file,
        done_file,
    ) = neb_result_paths()

    os.makedirs(
        folder_path,
        exist_ok=True,
    )

    os.makedirs(
        figure_path,
        exist_ok=True,
    )

    if (
        os.path.exists(done_file)
        and os.path.exists(result_file)
        and not FORCE_RERUN
    ):
        print(
            "[SKIP] ASE NEB ODE 已完成："
        )
        print(result_file)
        return

    print("\n" + "=" * 80)
    print("Alanine MFEP — ASE NEB + adaptive ODE")
    print("=" * 80)
    print(f"N internal images : {N}")
    print(f"ASE default k     : {ASE_DEFAULT_K}")
    print(f"ASE force cap     : {ASE_FORCE_CAP}")
    print("NEB method        : improvedtangent")
    print("Tangent override  : central difference, same as IVM")
    print("Optimizer         : NEBOptimizer(method='ode')")
    print(f"Maximum ODE steps : {MAX_NEB_STEPS}")
    print(f"fmax tolerance    : {NEB_FMAX_TOL}")
    print("=" * 80)

    images = build_neb_images()

    # 不传 k，使用 ASE 默认 k=0.1
    neb = NEB(
        images,
        climb=False,
        parallel=False,
        method="improvedtangent",
        allow_shared_calculator=False,
    )

    # 中心差分 tangent，与 IVM 一致
    install_central_difference_improved_tangent(
        neb
    )

    optimizer = VerboseNEBOptimizer(
        neb,
        method="ode",
        logfile=None,
        trajectory=None,
        verbose=1,
    )

    # ========================================================
    # 保存序列
    # ========================================================

    # 主要横坐标：
    # 第一个内部 image 的调用次数减去初始调用
    count_seq = []

    # ODE accepted step
    optimizer_step_seq = []

    # 全部内部 images 的调用总数
    total_image_md_calls_seq = []

    # 兼容原变量名
    image_md_calls_seq = []

    # 第一个内部 image 的原始调用次数，包含初始路径
    first_image_force_calls_raw_seq = []

    # 第一个内部 image 调用次数减去初始调用
    first_image_force_calls_seq = []

    # 每个保存状态下，所有内部 images 各自的调用次数
    all_image_force_calls_seq = []

    # GLOBAL/N，仅作为交叉检查
    path_gradient_sweeps_raw_seq = []

    # 完整路径
    images_seq = []

    # 两个与 IVM 一致的指标
    I_perp_seq = []
    I_perp_euclid_arc_seq = []

    # 诊断
    fmaxs = []
    fmax_neb_physical_seq = []
    neb_total_fmax_seq = []

    # Reference oracle 数据
    M_ref_seq = []
    F_ref_seq = []
    Mg_ref_seq = []

    # 实际传给 ASE 的截断后 M grad F
    Mg_ase_ref_seq = []

    # force cap 信息
    force_scale_seq = []
    force_capped_seq = []
    raw_Mg_norm_seq = []
    ase_Mg_norm_seq = []

    # 投影量
    perpG_Mg_seq = []
    perpE_Mg_seq = []

    # 积分权重
    node_weights_G_seq = []
    node_weights_E_seq = []

    # G 投影矩阵
    P_perp_G_seq = []

    # 防止同一个状态重复保存
    last_recorded_signature = {
        "step": None,
        "count": None,
    }

    start_time = time.perf_counter()

    # ========================================================
    # 保存当前已有序列
    # ========================================================
    def save_current_npz():
        np.savez(
            result_file,

            # 横坐标和调用次数
            count_seq=np.asarray(
                count_seq,
                dtype=int,
            ),

            optimizer_step_seq=np.asarray(
                optimizer_step_seq,
                dtype=int,
            ),

            total_image_md_calls_seq=np.asarray(
                total_image_md_calls_seq,
                dtype=int,
            ),

            image_md_calls_seq=np.asarray(
                image_md_calls_seq,
                dtype=int,
            ),

            first_image_force_calls_raw_seq=np.asarray(
                first_image_force_calls_raw_seq,
                dtype=int,
            ),

            first_image_force_calls_seq=np.asarray(
                first_image_force_calls_seq,
                dtype=int,
            ),

            all_image_force_calls_seq=np.asarray(
                all_image_force_calls_seq,
                dtype=int,
            ),

            path_gradient_sweeps_raw_seq=np.asarray(
                path_gradient_sweeps_raw_seq,
                dtype=float,
            ),

            # 路径
            images_seq=np.asarray(
                images_seq,
                dtype=float,
            ),

            # 两个指标
            I_perp_seq=np.asarray(
                I_perp_seq,
                dtype=float,
            ),

            I_perp_euclid_arc_seq=np.asarray(
                I_perp_euclid_arc_seq,
                dtype=float,
            ),

            # fmax
            fmaxs=np.asarray(
                fmaxs,
                dtype=float,
            ),

            fmax_neb_physical_seq=np.asarray(
                fmax_neb_physical_seq,
                dtype=float,
            ),

            neb_total_fmax_seq=np.asarray(
                neb_total_fmax_seq,
                dtype=float,
            ),

            # Reference M、F、M F
            M_ref_seq=np.asarray(
                M_ref_seq,
                dtype=float,
            ),

            F_ref_seq=np.asarray(
                F_ref_seq,
                dtype=float,
            ),

            Mg_ref_seq=np.asarray(
                Mg_ref_seq,
                dtype=float,
            ),

            Mg_ase_ref_seq=np.asarray(
                Mg_ase_ref_seq,
                dtype=float,
            ),

            # Force cap
            force_scale_seq=np.asarray(
                force_scale_seq,
                dtype=float,
            ),

            force_capped_seq=np.asarray(
                force_capped_seq,
                dtype=bool,
            ),

            raw_Mg_norm_seq=np.asarray(
                raw_Mg_norm_seq,
                dtype=float,
            ),

            ase_Mg_norm_seq=np.asarray(
                ase_Mg_norm_seq,
                dtype=float,
            ),

            # residual
            perpG_Mg_seq=np.asarray(
                perpG_Mg_seq,
                dtype=float,
            ),

            perpE_Mg_seq=np.asarray(
                perpE_Mg_seq,
                dtype=float,
            ),

            # weights/projector
            node_weights_G_seq=np.asarray(
                node_weights_G_seq,
                dtype=float,
            ),

            node_weights_E_seq=np.asarray(
                node_weights_E_seq,
                dtype=float,
            ),

            P_perp_G_seq=np.asarray(
                P_perp_G_seq,
                dtype=float,
            ),

            # Metadata
            k_spring=np.asarray(
                [ASE_DEFAULT_K],
                dtype=float,
            ),

            force_cap=np.asarray(
                [ASE_FORCE_CAP],
                dtype=float,
            ),

            optimizer_method=np.asarray(
                ["ode"],
                dtype=str,
            ),

            neb_method=np.asarray(
                ["improvedtangent_central_difference"],
                dtype=str,
            ),

            count_definition=np.asarray(
                [
                    "first interior image M_gradient calls minus 1"
                ],
                dtype=str,
            ),

            N_internal=np.asarray(
                [N],
                dtype=int,
            ),

            endpoints=np.stack(
                [C1, C2],
                axis=0,
            ),
        )

    # ========================================================
    # 记录当前状态
    # ========================================================
    def record_state(step_index=None):
        if step_index is None:
            step_index = int(
                optimizer.nsteps
            )
        else:
            step_index = int(
                step_index
            )

        neb_forces = neb.get_forces()

        path_full = images_to_path(
            images
        )

        (
            M_all,
            F_all,
            Mg_all_cached,
        ) = collect_internal_oracle_data(
            images
        )

        (
            Mg_ase_all,
            force_scale_all,
            force_capped_all,
            raw_Mg_norm_all,
            ase_Mg_norm_all,
        ) = collect_internal_force_cap_data(
            images
        )

        metrics = compute_integrated_metrics(
            path_full,
            M_all,
            F_all,
        )

        (
            all_image_calls,
            first_image_calls_raw,
        ) = get_internal_image_md_calls(
            images
        )

        if not np.all(
            all_image_calls
            == first_image_calls_raw
        ):
            print(
                "[WARNING] Interior-image force-call counts "
                "are not identical:"
            )
            print(all_image_calls)

        # 初始路径调用为 1，对应 count=0
        count_value = max(
            first_image_calls_raw - 1,
            0,
        )

        path_sweeps_raw = (
            GLOBAL_IMAGE_MD_CALLS
            / float(N)
        )

        if not np.isclose(
            path_sweeps_raw,
            float(first_image_calls_raw),
            rtol=0.0,
            atol=1e-12,
        ):
            print(
                "[WARNING] GLOBAL_IMAGE_MD_CALLS/N differs "
                "from first-image calls:"
            )
            print(
                f"    GLOBAL_IMAGE_MD_CALLS/N = "
                f"{path_sweeps_raw}"
            )
            print(
                f"    first image md_calls    = "
                f"{first_image_calls_raw}"
            )

        if (
            last_recorded_signature["step"]
            == step_index
            and last_recorded_signature["count"]
            == count_value
        ):
            return

        neb_force_rows = np.asarray(
            neb_forces,
            dtype=float,
        ).reshape(-1, 3)

        neb_total_fmax = float(
            np.max(
                np.linalg.norm(
                    neb_force_rows,
                    axis=1,
                )
            )
        )

        count_seq.append(
            int(count_value)
        )

        optimizer_step_seq.append(
            int(step_index)
        )

        total_image_md_calls_seq.append(
            int(GLOBAL_IMAGE_MD_CALLS)
        )

        image_md_calls_seq.append(
            int(GLOBAL_IMAGE_MD_CALLS)
        )

        first_image_force_calls_raw_seq.append(
            int(first_image_calls_raw)
        )

        first_image_force_calls_seq.append(
            int(count_value)
        )

        all_image_force_calls_seq.append(
            all_image_calls.copy()
        )

        path_gradient_sweeps_raw_seq.append(
            float(path_sweeps_raw)
        )

        images_seq.append(
            path_full.copy()
        )

        I_perp_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_euclid_arc_seq.append(
            metrics["I_perp_E"]
        )

        fmaxs.append(
            metrics[
                "fmax_Gperp_Euclidean"
            ]
        )

        fmax_neb_physical_seq.append(
            metrics[
                "fmax_NEB_physical"
            ]
        )

        neb_total_fmax_seq.append(
            neb_total_fmax
        )

        M_ref_seq.append(
            M_all.copy()
        )

        F_ref_seq.append(
            F_all.copy()
        )

        Mg_ref_seq.append(
            metrics["Mg_all"].copy()
        )

        Mg_ase_ref_seq.append(
            Mg_ase_all.copy()
        )

        force_scale_seq.append(
            force_scale_all.copy()
        )

        force_capped_seq.append(
            force_capped_all.copy()
        )

        raw_Mg_norm_seq.append(
            raw_Mg_norm_all.copy()
        )

        ase_Mg_norm_seq.append(
            ase_Mg_norm_all.copy()
        )

        perpG_Mg_seq.append(
            metrics["perpG_Mg"].copy()
        )

        perpE_Mg_seq.append(
            metrics["perpE_Mg"].copy()
        )

        node_weights_G_seq.append(
            metrics[
                "node_weights_G"
            ].copy()
        )

        node_weights_E_seq.append(
            metrics[
                "node_weights_E"
            ].copy()
        )

        P_perp_G_seq.append(
            metrics[
                "P_perp_G_all"
            ].copy()
        )

        last_recorded_signature["step"] = (
            step_index
        )

        last_recorded_signature["count"] = (
            count_value
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        n_capped = int(
            np.sum(force_capped_all)
        )

        print(
            f"[NEB-ODE] "
            f"accepted_step={step_index:4d}  "
            f"count_seq={count_value:4d}  "
            f"I_perp_G={metrics['I_perp_G']:.6e}  "
            f"I_perp_E={metrics['I_perp_E']:.6e}  "
            f"fmax_Gperp_E="
            f"{metrics['fmax_Gperp_Euclidean']:.6e}  "
            f"fmax_NEB_physical="
            f"{metrics['fmax_NEB_physical']:.6e}  "
            f"ASE_total_fmax={neb_total_fmax:.6e}  "
            f"first_image_calls={first_image_calls_raw}  "
            f"total_image_calls={GLOBAL_IMAGE_MD_CALLS}  "
            f"capped_images={n_capped}/{N}  "
            f"elapsed={elapsed:.1f}s"
        )

        if (
            SAVE_FIGS
            and (
                step_index == 0
                or step_index % SAVE_FIG_EVERY == 0
                or step_index == MAX_NEB_STEPS
            )
        ):
            try:
                fig_save(
                    step_index,
                    path_full[1:-1],
                    path_full[1:-1],
                    None,
                    path_full[1:-1],
                    xlim=xlim,
                    ylim=ylim,

                    # G-normal 原始 M grad F
                    force1=-metrics[
                        "perpG_Mg"
                    ],

                    # 原始 grad F
                    force2=-F_all,

                    # Euclidean-normal 原始 M grad F
                    force3=-metrics[
                        "perpE_Mg"
                    ],

                    figure_path=figure_path,
                )

            except Exception as exc:
                print(
                    f"[WARNING] fig_save failed "
                    f"at accepted step {step_index}: {exc}"
                )

            plt.close("all")

        if SAVE_EVERY_STATE:
            save_current_npz()

    # 保存初始路径
    record_state(
        step_index=0
    )

    # ASE observer 在 nsteps 增加前调用
    def record_after_accepted_step():
        record_state(
            step_index=optimizer.nsteps + 1
        )

    optimizer.attach(
        record_after_accepted_step,
        interval=1,
    )

    # 执行 ODE NEB
    converged = optimizer.run(
        fmax=NEB_FMAX_TOL,
        steps=MAX_NEB_STEPS,
    )

    # 最终状态
    record_state(
        step_index=optimizer.nsteps
    )

    save_current_npz()

    elapsed_total = (
        time.perf_counter()
        - start_time
    )

    (
        final_all_image_calls,
        final_first_image_calls,
    ) = get_internal_image_md_calls(
        images
    )

    final_comparison_count = max(
        final_first_image_calls - 1,
        0,
    )

    with open(
        done_file,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            "completed\n"
            f"converged={converged}\n"
            f"optimizer=ode\n"
            f"k={ASE_DEFAULT_K}\n"
            f"force_cap={ASE_FORCE_CAP}\n"
            f"accepted_steps={optimizer.nsteps}\n"
            f"first_image_force_calls="
            f"{final_first_image_calls}\n"
            f"comparison_count="
            f"{final_comparison_count}\n"
            f"total_image_md_calls="
            f"{GLOBAL_IMAGE_MD_CALLS}\n"
            f"all_image_force_calls="
            f"{final_all_image_calls.tolist()}\n"
            f"elapsed_seconds={elapsed_total}\n"
        )

    print("\n" + "=" * 80)
    print("ASE NEB ODE finished")
    print("=" * 80)
    print(f"Converged               : {converged}")
    print(f"Accepted ODE steps      : {optimizer.nsteps}")
    print(
        f"First-image force calls : "
        f"{final_first_image_calls}"
    )
    print(
        f"Comparison count        : "
        f"{final_comparison_count}"
    )
    print(
        f"Total image MD calls    : "
        f"{GLOBAL_IMAGE_MD_CALLS}"
    )
    print(
        f"All image call counts   : "
        f"{final_all_image_calls}"
    )
    print(f"Elapsed time (seconds)  : {elapsed_total:.1f}")
    print(f"Result file             : {result_file}")
    print(f"Figure folder           : {figure_path}")
    print("=" * 80)


# ============================================================
# 主程序
# ============================================================
def main():
    run_neb_experiment()


if __name__ == "__main__":
    main()


Alanine MFEP — ASE NEB + adaptive ODE
N internal images : 20
ASE default k     : 0.1
ASE force cap     : 500.0
NEB method        : improvedtangent
Tangent override  : central difference, same as IVM
Optimizer         : NEBOptimizer(method='ode')
Maximum ODE steps : 200
fmax tolerance    : 1e-08
phi_-145.97816739916388_psi_154.21681579813415/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-145.97816739916388_psi_154.21681579813415/PROJ: discarding 125/2501 frames (5.0%)
phi_-142.49077940130658_psi_149.34551932018863/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-142.49077940130658_psi_149.34551932018863/PROJ: discarding 125/2501 frames (5.0%)
phi_-139.00339140344929_psi_144.47422284224314/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-139.00339140344929_psi_144.47422284224314/PROJ: discarding 125/2501 frames (5.0%)
phi_-135.516003405592_psi_139.60292636429764/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-135.516003405592_psi_139.60292636429764

KeyboardInterrupt: 